1. Install Dependencies

In [ ]:
!pip install transformers datasets seqeval -q

2. Import Libraries

In [ ]:
import numpy as np
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForTokenClassification
from transformers import TrainingArguments, Trainer
from seqeval.metrics import classification_report

In [ ]:
from google.colab import files

uploaded = files.upload()

for fn in uploaded.keys():
  print(f'User uploaded file "{fn}" with length {len(uploaded[fn])} bytes')

3. Load Dataset (Manual JSON)

In [ ]:
dataset = load_dataset(
    "json",
    data_files={
        "train": "train.json",
        "validation": "valid.json",
        "test": "test.json"
    }
)

train_data = dataset["train"]
val_data = dataset["validation"]
test_data = dataset["test"]

print(train_data[0])

4. Define Labels

In [ ]:
label_list = [
    "O",
    "B-PER",
    "I-PER",
    "B-ORG",
    "I-ORG",
    "B-LOC",
    "I-LOC",
    "B-MISC",
    "I-MISC"
]

id2label = {i: label for i, label in enumerate(label_list)}
label2id = {label: i for i, label in enumerate(label_list)}

5. Load Tokenizer

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")

6. Tokenization + Label Alignment

In [ ]:
def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True
    )

    labels = []

    for i, label in enumerate(examples["tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []

        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])
            else:
                label_ids.append(label[word_idx])

            previous_word_idx = word_idx

        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

7. Apply Tokenization

In [ ]:
train_data = train_data.map(tokenize_and_align_labels, batched=True)
val_data = val_data.map(tokenize_and_align_labels, batched=True)
test_data = test_data.map(tokenize_and_align_labels, batched=True)

8. Load Model

In [ ]:
model = AutoModelForTokenClassification.from_pretrained(
    "bert-base-cased",
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id
)

9. Data Collator

In [ ]:
data_collator = DataCollatorForTokenClassification(tokenizer)

10. Training Arguments

In [ ]:
training_args = TrainingArguments(
    output_dir="./results",
    do_eval=True,
    logging_steps=100,
    learning_rate=2e-5,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    num_train_epochs=1,
    save_strategy="epoch"
)

11. Metrics

In [ ]:
def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_predictions = []
    true_labels = []

    for pred, lab in zip(predictions, labels):
        cur_preds = []
        cur_labels = []

        for p_, l_ in zip(pred, lab):
            if l_ != -100:
                cur_preds.append(id2label[p_])
                cur_labels.append(id2label[l_])

        true_predictions.append(cur_preds)
        true_labels.append(cur_labels)

    from seqeval.metrics import classification_report
    print(classification_report(true_labels, true_predictions))

    return {}

12. Trainer

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_data,
    eval_dataset=val_data,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

13. Train Model

In [ ]:
trainer.train()

14. Evaluate Model

In [ ]:
trainer.evaluate()

15. Save Model

In [ ]:
trainer.save_model("./ner-bert-model")
tokenizer.save_pretrained("./ner-bert-model")